In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"

# Convierte los notebooks ya existentes a modulos .py importables.
# train_pipeline.ipynb debe haberse ejecutado antes al menos una vez,
# ya que este notebook consume sus resultados (summary_statistics.json,
# all_runs.csv y los pesos .pt de cada corrida).
!jupyter nbconvert --to python "$PROJECT_DIR/preprocessing.ipynb"
!jupyter nbconvert --to python "$PROJECT_DIR/models.ipynb"
!jupyter nbconvert --to python "$PROJECT_DIR/train_pipeline.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.ipynb to python
[NbConvertApp] Writing 5391 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.py
[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/models.ipynb to python
[NbConvertApp] Writing 3362 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/models.py
[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/train_pipeline.ipynb to python
[NbConvertApp] Writing 15193 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/train_pipeline.py


In [26]:
import sys

PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/Trabajo_Cualitativo', '/content/drive/MyDrive/Trabajo_Cualitativo']


In [27]:
!pip install mlflow
import os
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import mlflow

from preprocessing import prepare_dataset
from models import build_model
from train_pipeline import (
    RESULTS_DIR,
    DATA_PATH,
    SPLIT_SEED,
    DEVICE,
    MLFLOW_DB_PATH,
    MLFLOW_EXPERIMENT,
    to_tensor,
)

!pip install mlflow

In [28]:
# --------------------------------------------------------------------- #
# Configuracion (rutas derivadas de RESULTS_DIR ya definido en
# train_pipeline.py; nada se retipea aqui)
# --------------------------------------------------------------------- #
SUMMARY_JSON_PATH = f"{RESULTS_DIR}/summary_statistics.json"
ALL_RUNS_PATH = f"{RESULTS_DIR}/all_runs.csv"
PLAYBOOK_DIR = f"{RESULTS_DIR}/playbook"
os.makedirs(PLAYBOOK_DIR, exist_ok=True)

# Metrica(s) usadas para elegir el modelo campeon. Son nombres de
# columna del cuadro comparativo (summary_statistics), configurables
# sin tocar la logica de seleccion.
PRIMARY_METRIC = "f1_mean"
STD_TIEBREAKER = "f1_std"
SECONDARY_METRIC = "roc_auc_mean"

# Percentiles usados para separar severidad Alta/Media/Baja dentro de
# los flujos marcados como ataque, calculados sobre la distribucion de
# score del propio lote evaluado (no son umbrales fijos de negocio).
SEVERITY_PERCENTILES = (0.33, 0.66)

# Catalogo de acciones de playbook por nivel de severidad. Esto es
# conocimiento de dominio (procedimiento del SOC), no un resultado del
# experimento; se mantiene como configuracion editable y separada de la
# logica de seleccion/inferencia.
PLAYBOOK_CATALOG = {
    "Alto": {
        "sla_minutos": 15,
        "acciones": [
            "Aislar/poner en cuarentena el host o flujo de red afectado (NAC/EDR)",
            "Bloquear IP y puerto de origen en el firewall perimetral",
            "Escalar de inmediato a analista SOC Nivel 2",
            "Abrir ticket de incidente Severidad 1",
        ],
    },
    "Medio": {
        "sla_minutos": 60,
        "acciones": [
            "Marcar el flujo para revision prioritaria por analista SOC Nivel 1",
            "Incrementar temporalmente el nivel de logging/captura del segmento",
            "Abrir ticket de incidente Severidad 2",
        ],
    },
    "Bajo": {
        "sla_minutos": 240,
        "acciones": [
            "Encolar para revision batch (no bloqueante)",
            "Registrar en watchlist para correlacion con eventos futuros",
        ],
    },
}

In [29]:
# --------------------------------------------------------------------- #
# Seleccion del modelo y la corrida campeon (100% a partir de los
# archivos generados por train_pipeline.py)
# --------------------------------------------------------------------- #
def load_training_results():
    """Carga el cuadro comparativo y el detalle por corrida.

    Falla explicitamente si train_pipeline.run_all() aun no se ha
    ejecutado, en vez de asumir valores por defecto.
    """
    if not (os.path.exists(SUMMARY_JSON_PATH) and os.path.exists(ALL_RUNS_PATH)):
        raise FileNotFoundError(
            "No se encontraron los resultados del entrenamiento en "
            f"{RESULTS_DIR}. Ejecuta primero train_pipeline.run_all()."
        )

    with open(SUMMARY_JSON_PATH, "r", encoding="utf-8") as file:
        summary_records = json.load(file)

    summary_df = pd.DataFrame(summary_records)
    all_runs_df = pd.read_csv(ALL_RUNS_PATH)

    return summary_df, all_runs_df


def select_champion_model(
    summary_df: pd.DataFrame,
    primary_metric: str = PRIMARY_METRIC,
    std_tiebreaker: str = STD_TIEBREAKER,
    secondary_metric: str = SECONDARY_METRIC,
) -> str:
    """Elige el modelo campeon del cuadro estadistico comparativo.

    Ordena por la metrica principal (desc), desempata por menor
    desviacion estandar y luego por la metrica secundaria (desc). No se
    asume de antemano cual modelo gana: el resultado depende por
    completo de los valores que train_pipeline.py haya calculado.
    """
    ranked = summary_df.sort_values(
        by=[primary_metric, std_tiebreaker, secondary_metric],
        ascending=[False, True, False],
    ).reset_index(drop=True)

    return ranked.loc[0, "Modelo"]


def select_champion_run(all_runs_df: pd.DataFrame, champion_model: str) -> pd.Series:
    """Dentro del modelo campeon, elige la corrida (semilla) con mejor F1."""
    model_runs = all_runs_df[all_runs_df["model"] == champion_model]

    if model_runs.empty:
        raise ValueError(f"No hay corridas registradas para el modelo {champion_model}")

    return model_runs.sort_values(by="f1", ascending=False).iloc[0]

In [30]:
# --------------------------------------------------------------------- #
# Carga del modelo campeon e inferencia sobre el lote evaluado
# --------------------------------------------------------------------- #
def load_champion_model(champion_model: str, seed: int, n_features: int):
    """Reconstruye la arquitectura (models.py) y carga los pesos .pt
    guardados por train_pipeline.py para esa corrida especifica."""
    run_name = f"{champion_model}_seed{seed}"
    artifact_path = f"{RESULTS_DIR}/model_{run_name}.pt"

    if not os.path.exists(artifact_path):
        raise FileNotFoundError(
            f"No se encontro el artefacto de la corrida campeon: {artifact_path}"
        )

    model = build_model(champion_model, n_features).to(DEVICE)
    state_dict = torch.load(artifact_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    return model, artifact_path, run_name


def score_batch(model, champion_model: str, X: np.ndarray, champion_row: pd.Series) -> np.ndarray:
    """Devuelve el score orientado a la clase positiva (1 = ataque),
    replicando exactamente la logica de evaluacion usada por
    train_pipeline.py para cada tipo de arquitectura."""
    model.eval()

    with torch.no_grad():
        if champion_model == "Autoencoder":
            X_t = to_tensor(X)
            recon = model(X_t)
            err = torch.mean((recon - X_t) ** 2, dim=1).cpu().numpy()
            score_dominant = -err

            dominant_label = int(champion_row["dominant_label_trained_on"])
            if dominant_label == 1:
                return score_dominant
            return -score_dominant

        logits = model(to_tensor(X))
        return torch.sigmoid(logits).cpu().numpy()


def assign_risk_tiers(scores: np.ndarray, y_pred: np.ndarray):
    """Separa los flujos marcados como ataque en Alto/Medio/Bajo usando
    percentiles calculados sobre el propio lote evaluado (no umbrales
    fijos de negocio)."""
    flagged_scores = scores[y_pred == 1]
    tiers = np.full(len(scores), "Benigno", dtype=object)

    if len(flagged_scores) == 0:
        return tiers, {}

    q_low, q_high = np.quantile(flagged_scores, list(SEVERITY_PERCENTILES))

    for i in range(len(scores)):
        if y_pred[i] == 0:
            continue
        if scores[i] >= q_high:
            tiers[i] = "Alto"
        elif scores[i] >= q_low:
            tiers[i] = "Medio"
        else:
            tiers[i] = "Bajo"

    thresholds_used = {
        f"percentil_{int(SEVERITY_PERCENTILES[0] * 100)}": float(q_low),
        f"percentil_{int(SEVERITY_PERCENTILES[1] * 100)}": float(q_high),
    }
    return tiers, thresholds_used

In [31]:
# --------------------------------------------------------------------- #
# Construccion y versionado de la actualizacion del playbook
# --------------------------------------------------------------------- #
def build_playbook_update(data, champion_model, champion_row, tiers, thresholds_used, y_pred) -> dict:
    tier_counts = pd.Series(tiers).value_counts().to_dict()

    metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
    metricas_modelo_campeon = {
        col: float(champion_row[col]) for col in metric_cols if col in champion_row
    }

    return {
        "generado_en_utc": datetime.now(timezone.utc).isoformat(),
        "modelo_campeon": champion_model,
        "corrida_seleccionada": f"{champion_model}_seed{int(champion_row['seed'])}",
        "criterio_seleccion": {
            "metrica_principal": PRIMARY_METRIC,
            "desempate_std": STD_TIEBREAKER,
            "metrica_secundaria": SECONDARY_METRIC,
        },
        "metricas_modelo_campeon": metricas_modelo_campeon,
        "umbral_decision": float(champion_row["threshold"]),
        "mapeo_target": data.get("target_mapping"),
        "labels_ataque_conocidas_en_entrenamiento": data.get("detected_attack_labels"),
        "lote_evaluado": {
            "n_flujos": int(len(y_pred)),
            "n_flujos_marcados_ataque": int(y_pred.sum()),
            "tasa_deteccion": float(y_pred.mean()),
        },
        "umbrales_severidad_percentiles": thresholds_used,
        "distribucion_por_nivel_severidad": {k: int(v) for k, v in tier_counts.items()},
        "acciones_playbook_por_nivel": PLAYBOOK_CATALOG,
    }


def write_playbook_version(update: dict):
    """Escribe una version timestamped del playbook, actualiza el
    puntero playbook_current.json y agrega una fila al changelog."""
    version_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    version_path = f"{PLAYBOOK_DIR}/playbook_update_{version_ts}.json"
    current_path = f"{PLAYBOOK_DIR}/playbook_current.json"
    changelog_path = f"{PLAYBOOK_DIR}/playbook_changelog.csv"

    with open(version_path, "w", encoding="utf-8") as file:
        json.dump(update, file, indent=2, ensure_ascii=False)

    with open(current_path, "w", encoding="utf-8") as file:
        json.dump(update, file, indent=2, ensure_ascii=False)

    changelog_row = pd.DataFrame([{
        "version_ts": version_ts,
        "modelo_campeon": update["modelo_campeon"],
        "corrida_seleccionada": update["corrida_seleccionada"],
        "f1": update["metricas_modelo_campeon"].get("f1"),
        "n_flujos_marcados_ataque": update["lote_evaluado"]["n_flujos_marcados_ataque"],
        "archivo": version_path,
    }])

    if os.path.exists(changelog_path):
        changelog_row.to_csv(changelog_path, mode="a", header=False, index=False)
    else:
        changelog_row.to_csv(changelog_path, index=False)

    return version_path, current_path, changelog_path

In [32]:
# --------------------------------------------------------------------- #
# Orquestacion principal
# --------------------------------------------------------------------- #
def run_playbook_update():
    summary_df, all_runs_df = load_training_results()

    champion_model = select_champion_model(summary_df)
    champion_row = select_champion_run(all_runs_df, champion_model)

    print(f"Modelo campeon (seleccionado por {PRIMARY_METRIC}): {champion_model}")
    print(
        f"Corrida seleccionada: seed={int(champion_row['seed'])} "
        f"| F1={champion_row['f1']:.4f} | umbral={champion_row['threshold']:.4f}"
    )

    data = prepare_dataset(DATA_PATH, random_state=SPLIT_SEED)

    model, artifact_path, run_name = load_champion_model(
        champion_model, int(champion_row["seed"]), data["n_features"]
    )

    X_eval = data["X_test"]
    scores = score_batch(model, champion_model, X_eval, champion_row)
    y_pred = (scores >= float(champion_row["threshold"])).astype(int)

    tiers, thresholds_used = assign_risk_tiers(scores, y_pred)

    update = build_playbook_update(data, champion_model, champion_row, tiers, thresholds_used, y_pred)
    version_path, current_path, changelog_path = write_playbook_version(update)

    mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name=f"playbook_update_{run_name}"):
        mlflow.log_params({
            "modelo_campeon": champion_model,
            "corrida_seleccionada": run_name,
            "metrica_seleccion": PRIMARY_METRIC,
            "umbral_decision": float(champion_row["threshold"]),
        })
        mlflow.log_metrics({
            "n_flujos_marcados_ataque": int(y_pred.sum()),
            "tasa_deteccion": float(y_pred.mean()),
        })
        mlflow.log_artifact(version_path)
        mlflow.log_artifact(changelog_path)
        mlflow.log_artifact(artifact_path)

    print("\n=== ACTUALIZACION DE PLAYBOOK GENERADA ===")
    print(f"Modelo campeon: {champion_model} (corrida {run_name})")
    print(f"Flujos evaluados: {len(y_pred)} | Marcados como ataque: {int(y_pred.sum())}")
    print("Distribucion por nivel de severidad:", update["distribucion_por_nivel_severidad"])
    print("\nArchivos generados:")
    print("-", version_path)
    print("-", current_path)
    print("-", changelog_path)

    return update


if __name__ == "__main__":
    run_playbook_update()

Modelo campeon (seleccionado por f1_mean): MLP
Corrida seleccionada: seed=43 | F1=0.9975 | umbral=0.9991

=== ACTUALIZACION DE PLAYBOOK GENERADA ===
Modelo campeon: MLP (corrida MLP_seed43)
Flujos evaluados: 208110 | Marcados como ataque: 10460
Distribucion por nivel de severidad: {'Benigno': 197650, 'Alto': 3752, 'Bajo': 3447, 'Medio': 3261}

Archivos generados:
- /content/drive/MyDrive/Trabajo_Cualitativo/results/playbook/playbook_update_20260728T050718Z.json
- /content/drive/MyDrive/Trabajo_Cualitativo/results/playbook/playbook_current.json
- /content/drive/MyDrive/Trabajo_Cualitativo/results/playbook/playbook_changelog.csv
